In [1]:
from pathlib import Path
import pandas as pd
import pickle
import torch
import re
from collections import Counter

PROJECT_ROOT = Path.cwd().parent
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

PAD, SOS, EOS, UNK = "<pad>", "<sos>", "<eos>", "<unk>"

def tokenize(s):
    return re.findall(r"[a-z]+", s.lower())

class Vocab:
    def __init__(self, token_lists, min_freq=5):
        counts = Counter(t for toks in token_lists for t in toks)
        self.itos = [PAD, SOS, EOS, UNK] + sorted(w for w, c in counts.items() if c >= min_freq)
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def encode(self, toks):
        return [self.stoi[SOS]] + [self.stoi.get(t, self.stoi[UNK]) for t in toks] + [self.stoi[EOS]]

    def decode(self, ids):
        return " ".join(self.itos[i] for i in ids if i not in (0, 1, 2))

train_df = pd.read_csv(ARTIFACTS_DIR / "train.csv")
val_df   = pd.read_csv(ARTIFACTS_DIR / "val.csv")
test_df  = pd.read_csv(ARTIFACTS_DIR / "test.csv")

with open(ARTIFACTS_DIR / "vocab.pkl", "rb") as f:
    vocab = pickle.load(f)

with open(ARTIFACTS_DIR / "features.pkl", "rb") as f:
    feat_data = pickle.load(f)

filenames_list = feat_data["filenames"]
feature_matrix = feat_data["matrix"]              # (8091, 2048)
fname_to_idx = {f: i for i, f in enumerate(filenames_list)}   # filename -> row index

print(train_df.shape, val_df.shape, test_df.shape, len(vocab.itos), feature_matrix.shape)

(32360, 2) (4045, 2) (4050, 2) 2652 torch.Size([8091, 2048])


In [2]:
from torch.utils.data import Dataset

class CaptionDataset(Dataset):
    def __init__(self, df, feature_matrix, fname_to_idx, vocab):
        self.df = df.reset_index(drop=True)
        self.feature_matrix = feature_matrix
        self.fname_to_idx = fname_to_idx
        self.vocab = vocab

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = row["image"]
        caption = row["caption"]

        feat_idx = self.fname_to_idx[fname]
        feature = self.feature_matrix[feat_idx].float()   # back to float32 for training

        tokens = tokenize(caption)
        ids = self.vocab.encode(tokens)                   # [<sos>, w1, w2, ..., <eos>]

        return feature, torch.tensor(ids, dtype=torch.long)

train_dataset = CaptionDataset(train_df, feature_matrix, fname_to_idx, vocab)
val_dataset   = CaptionDataset(val_df, feature_matrix, fname_to_idx, vocab)
test_dataset  = CaptionDataset(test_df, feature_matrix, fname_to_idx, vocab)

print(len(train_dataset), len(val_dataset), len(test_dataset))

feat, ids = train_dataset[0]
print(feat.shape, feat.dtype)
print(ids.shape, ids)
print(vocab.decode(ids.tolist()))

32360 4045 4050
torch.Size([2048]) torch.float32
torch.Size([19]) tensor([   1,    4,  447, 1131,    4, 1652,  672, 1161,  480, 2507,    4, 1951,
        1492, 2178, 1131,   47,    3, 2577,    2])
a child in a pink dress is climbing up a set of stairs in an <unk> way


In [3]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    features, captions = zip(*batch)

    features = torch.stack(features)                      # (batch_size, 2048)

    lengths = torch.tensor([len(c) for c in captions])
    captions_padded = pad_sequence(captions, batch_first=True, padding_value=vocab.stoi[PAD])

    return features, captions_padded, lengths

from torch.utils.data import DataLoader

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

feat_batch, cap_batch, len_batch = next(iter(train_loader))
print(feat_batch.shape)
print(cap_batch.shape)
print(len_batch[:5])
print(cap_batch[0])

torch.Size([32, 2048])
torch.Size([32, 22])
tensor([10, 19, 16, 16, 12])
tensor([   1,    4,  653, 1161,  443, 1503,    4, 1385, 1691,    2,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0])


In [4]:
import torch.nn as nn

class DecoderLSTM(nn.Module):
    def __init__(self, feature_dim, embed_dim, hidden_dim, vocab_size, num_layers=1):
        super().__init__()
        self.init_h = nn.Linear(feature_dim, hidden_dim)   
        self.init_c = nn.Linear(feature_dim, hidden_dim)   

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, features, captions):
        h0 = self.init_h(features).unsqueeze(0)  
        c0 = self.init_c(features).unsqueeze(0)

        embedded = self.embedding(captions)        
        lstm_out, _ = self.lstm(embedded, (h0, c0)) 

        logits = self.fc_out(lstm_out)              
        return logits

In [5]:
model = DecoderLSTM(feature_dim=2048, embed_dim=256, hidden_dim=512, vocab_size=len(vocab.itos))
logits = model(feat_batch, cap_batch)
print(logits.shape)

torch.Size([32, 22, 2652])


In [6]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss(ignore_index=vocab.stoi[PAD])
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def train_one_batch(features, captions):
    features = features.to(device)
    captions = captions.to(device)

    inputs = captions[:, :-1]    
    targets = captions[:, 1:]   

    optimizer.zero_grad()
    logits = model(features, inputs)             

    loss = criterion(
        logits.reshape(-1, logits.size(-1)),      
        targets.reshape(-1)                        
    )

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
    optimizer.step()

    return loss.item()

loss = train_one_batch(feat_batch, cap_batch)
print(loss)

7.883275032043457


In [11]:
from tqdm import tqdm
def evaluate(loader):
    model.eval()
    total_loss = 0.0
    n_batches = 0

    with torch.no_grad():
        for features, captions, lengths in loader:
            features = features.to(device)
            captions = captions.to(device)

            inputs = captions[:, :-1]
            targets = captions[:, 1:]

            logits = model(features, inputs)
            loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

            total_loss += loss.item()
            n_batches += 1

    model.train()
    return total_loss / n_batches


def train_one_epoch(loader):
    model.train()
    total_loss = 0.0
    n_batches = 0

    for features, captions, lengths in tqdm(loader):
        loss = train_one_batch(features, captions)
        total_loss += loss
        n_batches += 1

    return total_loss / n_batches


NUM_EPOCHS = 10
best_val_loss = float("inf")

for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch(train_loader)
    val_loss = evaluate(val_loader)
    print(f"epoch {epoch+1}/{NUM_EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), ARTIFACTS_DIR / "best_model.pt")
        print(f"  -> saved new best model (val_loss={val_loss:.4f})")

100%|██████████| 1012/1012 [00:19<00:00, 51.36it/s]


epoch 1/10  train_loss=0.9669  val_loss=3.2399
  -> saved new best model (val_loss=3.2399)


100%|██████████| 1012/1012 [00:20<00:00, 49.13it/s]


epoch 2/10  train_loss=0.8299  val_loss=3.3637


100%|██████████| 1012/1012 [00:20<00:00, 49.30it/s]


...

epoch 5/10  train_loss=0.5809  val_loss=3.7485


100%|██████████| 1012/1012 [00:18<00:00, 53.56it/s]


epoch 6/10  train_loss=0.5294  val_loss=3.8525


 83%|████████▎ | 840/1012 [00:17<00:03, 48.92it/s]


KeyboardInterrupt: 

In [12]:
print(model)
for p in model.parameters():
    print(p.flatten()[:5])
    break

DecoderLSTM(
  (init_h): Linear(in_features=2048, out_features=512, bias=True)
  (init_c): Linear(in_features=2048, out_features=512, bias=True)
  (embedding): Embedding(2652, 256, padding_idx=0)
  (lstm): LSTM(256, 512, batch_first=True)
  (fc_out): Linear(in_features=512, out_features=2652, bias=True)
)
tensor([-0.1981, -0.0676,  0.0362,  0.0391, -0.3112], device='cuda:0',
       grad_fn=<SliceBackward0>)


In [13]:
print(id(model))
for p in model.parameters():
    print(p.flatten()[:5])
    break

2670805057312
tensor([-0.1981, -0.0676,  0.0362,  0.0391, -0.3112], device='cuda:0',
       grad_fn=<SliceBackward0>)
